## ❤️‍🩹 DVC를 사용한 데이터 변경 및 복구

머신러닝 및 데이터 과학 프로젝트에서 데이터는 정적이지 않으며, 새로운 인사이트가 나타나거나 수정이 이루어질 때 변합니다. 이러한 업데이트는 필요하지만 도전 과제를 야기할 수 있습니다: 이전 버전이 보존되었는지 어떻게 확인하시겠습니까? 문제를 해결하거나 결과를 비교하기 위해 이전 버전으로 롤백해야 한다면 어떻게 할까요?

**DVC(Data Version Control)**가 여기서 역할을 합니다. 데이터에 대한 모든 변경을 추적하고, 그러한 변경을 문서화하고, 필요할 때 이전 버전을 쉽게 복구할 수 있는 안전한 방법을 상상해 보세요. 저장소에 저장된 파일을 수정하거나 다양한 데이터 전처리 단계를 실험하든 DVC는 제어와 추적 가능성을 유지하도록 도와줍니다.

이 노트북에서는 다음에 중점을 둡니다:

1. S3에 저장된 Parquet 파일에 대한 변경 수행.
2. DVC를 사용하여 이러한 변경을 추적하고 문서화.
3. 필요할 때 이전 데이터 버전 복구.

마지막에는 DVC가 데이터 변경 관리 프로세스를 어떻게 간소화하는지, 그리고 이전 버전의 복구가 어떻게 단 몇 가지 명령으로 가능한지 보게 될 것입니다.

## 🐠 의존성 가져오기

먼저 노트북을 실행할 수 있도록 일부 의존성을 가져와야 합니다.

In [ ]:
# 의존성 가져오기
import boto3
import pyarrow
import pyarrow.parquet as pq
import pandas as pd
import os

## ✏️ S3의 Parquet 파일 수정

S3 버킷에 저장된 Parquet 파일이 있다고 가정합시다. 이제 최신 데이터로 업데이트해야 합니다. 필요한 경우 변경 사항을 추적하고 원본을 복원할 수 있도록 하면서 이를 어떻게 할 수 있을까요?

다음 계획입니다:
1. S3 버킷에서 Parquet 파일을 다운로드하고 열기.
2. 변경 수행(예: 새로운 데이터 추가).
3. 업데이트된 파일을 S3로 다시 저장하여 사용할 준비가 되고 적절하게 버전 관리되도록 합니다.

이러한 단계를 통해 모든 것이 정돈되고 추적하기 쉽도록 데이터를 업데이트할 수 있습니다.


In [ ]:
# 파일 다운로드
fs = pyarrow.fs.S3FileSystem(
        endpoint_override=os.environ.get('AWS_S3_ENDPOINT'),
        access_key=os.environ.get('AWS_ACCESS_KEY_ID'),
        secret_key=os.environ.get('AWS_SECRET_ACCESS_KEY')
    )

with fs.open_input_file('data/song_properties.parquet') as file:
    df = pd.read_parquet(file)

# 일부 변경 수행
df = pd.concat([df, df], ignore_index=True)

# 파일 업로드
pq.write_table(pyarrow.table(df), 'data/song_properties.parquet', filesystem=fs)

## 📦 DVC로 새로운 데이터 버전 생성

데이터를 수정한 후 **DVC(Data Version Control)**를 사용하여 변경 사항을 추적하는 것이 필수적입니다. `dvc import`, `dvc import-url` 또는 `dvc import-db`를 통해 가져온 파일이나 디렉토리의 경우 `dvc update`를 사용하여 데이터 소스의 최신 상태와 동기화합니다.

In [ ]:
# 새로운 변경 사항으로 버전 업데이트
!dvc update song_properties.parquet.dvc --to-remote

## 🛠️ Git에서 변경 사항 추적

데이터 버전을 코드와 연결하려면 Git에서 변경 사항을 커밋하세요.

In [ ]:
!git diff ../.dvc/config

In [ ]:
# Git에서 변경 사항 추적
!git add song_properties.parquet.dvc
!git commit -m "updated data"

## 🔄 이전 데이터 버전으로 되돌리기



이제 원하면 Git에서 이전 버전을 체크아웃하여 해당 Git과 함께 사용된 데이터 버전이 무엇인지 알 수 있습니다

In [ ]:
# 이전 dvc 파일로 되돌리기
!git checkout HEAD~1 song_properties.parquet.dvc

원본 파일을 다운로드하고 데이터 저장소로 푸시합니다(DVC를 통해 직접 푸시할 방법이 없습니다)

In [ ]:
!dvc pull
df = pd.read_parquet('song_properties.parquet', engine='pyarrow')
pq.write_table(pyarrow.table(df), 'data/song_properties.parquet', filesystem=fs)

## ✅ 복구된 데이터 복원 및 추적

이제 원본 데이터로 돌아가고 복구를 추적할 수 있습니다!

In [ ]:
# 복구된 데이터로 dvc를 다시 업데이트하고 버전 관리
!dvc update song_properties.parquet.dvc --to-remote
!git add song_properties.parquet.dvc
!git commit -m "reverted data"

참고: 원본 버전으로 되돌린 후 선택적으로 새 버전을 만들어 복구를 추적할 수 있습니다.

In [ ]:
# 마지막 커밋 3개를 모두 취소합시다
# 스포일러: 이 단계들을 자동화하니까요!
!git reset --hard HEAD~3

## 🎯 요약

이 워크플로우는 DVC가 데이터 버전 관리를 어떻게 지원하는지 보여줍니다:

* 데이터셋의 변경 사항을 수정하고 추적합니다.
* Git을 사용하여 데이터 및 코드 버전을 연결합니다.
* 필요할 때 이전 데이터셋 버전으로 되돌립니다.